<a href="https://colab.research.google.com/github/jackevansadl/chem2002/blob/main/C2/03_Crystal_Field_TM_Complexes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Part 3: Crystal Field Theory — Colour and Magnetism of Metal Complexes 🌈

Why is a solution of [Ti(H$_2$O)$_6$]$^{3+}$ **purple**, while [Cu(H$_2$O)$_6$]$^{2+}$ is **blue**? Why are some iron complexes magnetic and others not? The answer lies in the **d orbitals** of the transition-metal ion and how they respond to the surrounding **ligands**.

In an isolated metal ion the five *d* orbitals all have the same energy. Put the ion at the centre of an **octahedron** of six ligands and that changes: the *d* orbitals **split** into two groups. The size of the split, $\Delta_o$, controls the complex's **colour** and **magnetism**.

In this notebook you will:
* visualise the five *d* orbitals and see why they split,
* build the octahedral splitting diagram and fill it with electrons (**high-** vs **low-spin**),
* connect $\Delta_o$ to the **colour** you actually see.

> ⏱️ **Time guide:** about **75 minutes** (plus an optional ~5 min calculation at the end). Work through each part and answer the 📋 and 💭 prompts in your workbook.

### What you will do
1. Visualise the five **d orbitals** and see why they split in an octahedral field.
2. Fill the split orbitals and decide **high-spin vs low-spin**.
3. Connect the splitting $\Delta_o$ to **colour** and **magnetism**.

> 📚 **Lecture connection — Crystal Field vs Ligand Field Theory.** This workshop uses **Crystal Field Theory (CFT)**, which treats the ligands as simple **point negative charges**. It is a brilliant first model and gets the splitting pattern, colour and magnetism right. But it ignores that real metal–ligand bonds are partly **covalent**. In your CHEM 2002 lectures you will meet the more complete **Ligand Field Theory (LFT)**, which combines these ideas with the **molecular-orbital** approach from notebooks 1 and 2 — the metal *d* orbitals actually *overlap* with ligand orbitals. The good news: the same $t_{2g}/e_g$ splitting picture you build here carries straight over to LFT, just with a richer explanation of *why* it happens.

-----
## 📚 Crystal Field Theory in a nutshell

A transition-metal ion on its own has five *d* orbitals that all share the same energy (they are *degenerate*). Surround it with **ligands** — ions or molecules that donate a lone pair of electrons — and that changes. **Crystal Field Theory (CFT)** models each ligand as a small **point negative charge**. Electrons in *d* orbitals that point *towards* those charges are pushed up in energy; electrons in orbitals that point *between* them are pushed up much less.

In an **octahedral** complex (six ligands sitting on the $x, y, z$ axes) this splits the five *d* orbitals into a lower group of three (**$t_{2g}$**) and an upper group of two (**$e_g$**), separated by an energy gap called **$\Delta_o$** (the *octahedral crystal-field splitting*). Almost everything interesting about transition-metal complexes — their **colours**, their **magnetism**, even some of their **stability** — follows from the size of $\Delta_o$ and how the *d* electrons fill these two groups. That is exactly what you will explore below.

In [ ]:
!pip install -q pyscf py3Dmol matplotlib

In [ ]:
# Imports and helper functions
import numpy as np
import matplotlib.pyplot as plt
import py3Dmol
from pyscf import gto, dft
from pyscf.tools import cubegen

BOHR = 0.529177

def mol_to_xyz(mol):
    coords = mol.atom_coords() * BOHR
    lines = [str(mol.natm), ""]
    for i in range(mol.natm):
        x, y, z = coords[i]
        lines.append(f"{mol.atom_symbol(i)} {x:.4f} {y:.4f} {z:.4f}")
    return chr(10).join(lines)

def show_ao(mol, ao_index, isoval=0.04):
    """Render a single atomic orbital (basis function) by index."""
    coeff = np.zeros(mol.nao); coeff[ao_index] = 1.0
    cubegen.orbital(mol, "/tmp/orb.cube", coeff, nx=50, ny=50, nz=50)
    cube = open("/tmp/orb.cube").read()
    v = py3Dmol.view(width=350, height=300)
    v.addVolumetricData(cube, "cube", {"isoval":  isoval, "color": "blue", "opacity": 0.85})
    v.addVolumetricData(cube, "cube", {"isoval": -isoval, "color": "red",  "opacity": 0.85})
    v.zoomTo()
    return v.show()

-----
## 3.1 The Five *d* Orbitals

A transition metal has five *d* orbitals. They fall into **two shapes**:

* **Pointing *between* the axes:** d$_{xy}$, d$_{xz}$, d$_{yz}$  — these become the **$t_{2g}$** set.
* **Pointing *along* the axes:** d$_{z^2}$, d$_{x^2-y^2}$  — these become the **$e_g$** set.

Run the cell to view all five (blue = +, red = - lobes).

In [ ]:
# A single titanium atom -- view its five 3d orbitals
metal = gto.M(atom="Ti 0 0 0", basis="def2-svp", spin=2, verbose=0)
labels = metal.ao_labels()
d_orbitals = {l.split()[-1]: i for i, l in enumerate(labels) if "3d" in l}

for name, idx in d_orbitals.items():
    print("d orbital:", name)
    show_ao(metal, idx)

📖 **Which orbitals are which?** Look again at the five shapes:
* **d$_{z^2}$** and **d$_{x^2-y^2}$** have lobes pointing **along the axes** → these become the **$e_g$** set.
* **d$_{xy}$**, **d$_{xz}$**, **d$_{yz}$** have lobes pointing **between the axes** → these become the **$t_{2g}$** set.

In an octahedral complex the six ligands sit *on the axes*, so the $e_g$ orbitals point **straight at them**. Hold that thought for the next section.

-----
## 3.2 Why They Split: the Octahedral Field

In an octahedral complex the six ligands sit **on the x, y and z axes**. They are negatively charged (lone pairs pointing at the metal).

* The **$e_g$** orbitals (d$_{z^2}$, d$_{x^2-y^2}$) point **straight at the ligands** -> electrons there are repelled strongly -> **higher** energy.
* The **$t_{2g}$** orbitals (d$_{xy}$, d$_{xz}$, d$_{yz}$) point **between** the ligands -> less repulsion -> **lower** energy.

The energy gap between them is the **crystal-field splitting, $\Delta_o$**.

In [ ]:
# Draw the octahedral d-orbital splitting diagram
def draw_splitting(delta_o=1.0):
    fig, ax = plt.subplots(figsize=(5, 4))
    # barycentre at 0: t2g at -0.4, eg at +0.6
    for x in (-0.4, 0.1, 0.6):            # three t2g orbitals
        ax.hlines(-0.4 * delta_o, x, x + 0.35, color="#1f77b4", lw=3)
    for x in (0.0, 0.55):                 # two eg orbitals
        ax.hlines(0.6 * delta_o, x, x + 0.35, color="#d62728", lw=3)
    ax.annotate("", xy=(0.95, 0.6 * delta_o), xytext=(0.95, -0.4 * delta_o),
                arrowprops=dict(arrowstyle="<->", color="k"))
    ax.text(1.0, 0.1 * delta_o, r"$\Delta_o$", fontsize=14)
    ax.text(-0.5, -0.4 * delta_o - 0.18, r"$t_{2g}$ (d$_{xy}$, d$_{xz}$, d$_{yz}$)", color="#1f77b4")
    ax.text(-0.1, 0.6 * delta_o + 0.12, r"$e_g$ (d$_{z^2}$, d$_{x^2-y^2}$)", color="#d62728")
    ax.set_xlim(-0.7, 1.6); ax.set_ylim(-1, 1.2); ax.axis("off")
    ax.set_title("Octahedral crystal-field splitting")
    plt.show()

draw_splitting()

-----
## 3.3 Filling the Orbitals: High Spin vs Low Spin

Now we add the metal's *d* electrons. For **d$^4$ to d$^7$** there is a choice:

* If $\Delta_o$ is **small** (weak-field ligands): electrons spread out to avoid pairing -> **high spin** (more unpaired electrons).
* If $\Delta_o$ is **large** (strong-field ligands): it costs less to pair up in $t_{2g}$ than to climb to $e_g$ -> **low spin** (fewer unpaired electrons).

The deciding factor is $\Delta_o$ versus the **pairing energy** $P$. The function below works it out and also reports the **Crystal Field Stabilisation Energy (CFSE)**.

In [ ]:
# Fill the d orbitals of an octahedral complex

def fill_octahedral(n_d, low_spin):
    """Return (t2g electrons, eg electrons) for n_d d-electrons."""
    if low_spin:
        t2g = min(n_d, 6); eg = n_d - t2g
    else:  # high spin: singly fill all 5 orbitals first, then pair
        singles = min(n_d, 5)
        t2g = min(singles, 3); eg = min(max(singles - 3, 0), 2)
        extra = n_d - singles
        add_t = min(extra, 3); t2g += add_t; eg += extra - add_t
    return t2g, eg

def unpaired(t2g, eg):
    f = lambda n, norb: n if n <= norb else 2 * norb - n
    return f(t2g, 3) + f(eg, 2)

def analyse_complex(n_d, delta_o_cm, P_cm):
    low_spin = delta_o_cm > P_cm           # large splitting beats pairing cost
    t2g, eg = fill_octahedral(n_d, low_spin)
    unp = unpaired(t2g, eg)
    cfse = -0.4 * t2g + 0.6 * eg           # in units of Delta_o
    if abs(cfse) < 1e-9:
        cfse = 0.0
    spin = "LOW spin" if low_spin else "HIGH spin"
    print(f"d{n_d}: Delta_o={delta_o_cm} cm-1, P={P_cm} cm-1  ->  {spin}")
    print(f"   t2g^{t2g} eg^{eg}   unpaired e- = {unp}   CFSE = {cfse:+.1f} Delta_o")
    return t2g, eg, unp

In [ ]:
# Example: an Fe(III) complex (d5). Compare a weak-field and a strong-field case.
print("Weak-field ligand (e.g. H2O):")
analyse_complex(n_d=5, delta_o_cm=13700, P_cm=20000)
print()
print("Strong-field ligand (e.g. CN-):")
analyse_complex(n_d=5, delta_o_cm=35000, P_cm=20000)

📖 **The spectrochemical series and pairing energy.** Whether a complex is high- or low-spin is a tug-of-war between two energies:
* **$\Delta_o$** — the cost of promoting an electron up to $e_g$ instead of staying in $t_{2g}$.
* **$P$** — the *pairing energy*, the cost of forcing two electrons into the **same** orbital.

Ligands that produce a **large $\Delta_o$** (strong-field, e.g. CN⁻, CO) favour **low spin**; ligands with a **small $\Delta_o$** (weak-field, e.g. I⁻, Cl⁻) favour **high spin**. The **spectrochemical series** ranks ligands from weak to strong field — you'll use it in the reporting task.

-----
### 🔧 Hands-on Exercise 3.A — High spin across the d-block ⏱️ ~10 min

Run the loop to tabulate the number of **unpaired electrons** for every d-count from d$^1$ to d$^{10}$ in a **weak-field** (high-spin) complex. **Predict first:** which d-count will have the *most* unpaired electrons?

In [ ]:
# Weak-field (high-spin) octahedral complexes: Delta_o is small, so P wins
print("High-spin octahedral complexes (weak field):\n")
for n in range(1, 11):
    t2g, eg, unp = analyse_complex(n_d=n, delta_o_cm=10000, P_cm=20000)
    print()

-----
## 3.4 Colour: from $\Delta_o$ to What You See

A complex absorbs a photon when an electron jumps from $t_{2g}$ up to $e_g$ — a **d–d transition**. The energy of that jump is $\Delta_o$:

$$\Delta_o = \frac{hc}{\lambda}$$

So $\Delta_o$ fixes the **wavelength absorbed**. The colour you *see* is the **complementary** colour of the one absorbed.

In [ ]:
# From Delta_o to the colour you see
def colour_from_delta(delta_o_cm):
    lam_abs = 1e7 / delta_o_cm            # cm^-1  ->  nm
    E_eV = delta_o_cm * 1.23984e-4        # cm^-1  ->  eV
    wheel = [(400,430,"violet","yellow-green"), (430,490,"blue","orange"),
             (490,560,"green","red / purple"), (560,580,"yellow","violet"),
             (580,620,"orange","blue"), (620,750,"red","green")]
    absorbed, observed = "outside visible range", "-"
    for lo, hi, a, o in wheel:
        if lo <= lam_abs < hi:
            absorbed, observed = a, o; break
    print(f"Delta_o = {delta_o_cm} cm-1  =  {E_eV:.2f} eV")
    print(f"Absorbs light near {lam_abs:.0f} nm ({absorbed})")
    print(f"=> the complex appears {observed.upper()}")
    return lam_abs

# [Ti(H2O)6]3+ has Delta_o = 20300 cm-1
colour_from_delta(20300)

📖 **Why these colours are often pale.** A d–d transition (an electron jumping $t_{2g}\to e_g$) is technically "forbidden" by symmetry, so it absorbs only **weakly** — which is why many transition-metal solutions are *delicately* coloured rather than intense. The **energy** of the transition (set by $\Delta_o$) fixes the **hue**; the weak absorption sets the pale **intensity**.

-----
### 💭 Reflect
1. A **d$^0$** ion (e.g. Ti$^{4+}$) and a **d$^{10}$** ion (e.g. Zn$^{2+}$) are both usually **colourless**. Using the idea of a $t_{2g}\to e_g$ (d–d) transition, explain why. *(Hint: what has to be true for an electron to be able to jump?)*
2. Why does a **stronger-field** ligand tend to make a complex absorb at a **shorter** wavelength?

-----
### **📋 Reporting Task 3**

The **spectrochemical series** ranks ligands by the size of $\Delta_o$ they produce (weak field ... strong field):

$$\text{I}^- < \text{Cl}^- < \text{F}^- < \text{H}_2\text{O} < \text{NH}_3 < \text{CN}^-$$

Using the functions above, complete the following in your workbook:

1. For an **Fe(III) (d$^5$)** complex, use `analyse_complex` with a pairing energy of `P_cm = 20000`. Find the number of **unpaired electrons** for a weak-field ligand ($\Delta_o = 11000$) and a strong-field ligand ($\Delta_o = 33000$). Explain the difference in terms of high/low spin.
2. Use `colour_from_delta` to predict the colour of a complex with $\Delta_o = 17400$ cm$^{-1}$ (this is [Cr(H$_2$O)$_6$]$^{3+}$, a d$^3$ complex). What colour is it?
3. In one or two sentences, explain why strong-field ligands tend to give **larger** $\Delta_o$ and therefore absorb at **shorter** wavelengths.

In [ ]:
# Your answers here -- call analyse_complex(...) and colour_from_delta(...)

-----
### 🚀 Optional Bonus: a real quantum calculation of [Ti(H$_2$O)$_6$]$^{3+}$

Everything above used the *model* of crystal field theory. The cell below runs a genuine DFT calculation of the actual octahedral complex with PySCF. It takes around **30–60 seconds**. (Optional — not needed for the reporting task.)

In [ ]:
# (Optional, ~30-60 s) Build and run the real octahedral complex [Ti(H2O)6]3+
d = 2.03
axes = [(d,0,0),(-d,0,0),(0,d,0),(0,-d,0),(0,0,d),(0,0,-d)]
atoms = [("Ti", (0,0,0))]
for (x,y,z) in axes:
    r = np.array([x,y,z], float); u = r/np.linalg.norm(r)
    perp = np.cross(u, [0,0,1.0])
    if np.linalg.norm(perp) < 1e-3: perp = np.cross(u, [0,1.0,0])
    perp = perp/np.linalg.norm(perp)
    atoms.append(("O", tuple(r)))
    for s in (+1,-1):
        atoms.append(("H", tuple(r + u*0.6 + perp*0.78*s)))
atomstr = "; ".join(f"{a} {p[0]:.3f} {p[1]:.3f} {p[2]:.3f}" for a,p in atoms)

mol = gto.M(atom=atomstr, basis="def2-svp", charge=3, spin=1, verbose=0)
mf = dft.UKS(mol, xc="b3lyp"); mf.kernel()
print(f"Converged: {mf.converged}    Total energy: {mf.e_tot:.2f} Hartree")
print("This is a genuine quantum-chemistry calculation of a transition-metal complex!")

-----
> 📚 **Looking ahead:** everything here used **Crystal Field Theory** (ligands as point charges). In lectures you'll extend this to **Ligand Field Theory**, which brings in the covalent metal–ligand orbital overlap (the MO ideas from notebooks 1–2) for a fuller picture — while keeping the same $t_{2g}/e_g$ splitting at its core.

-----
## Congratulations! 🎉

You've completed the C2 workshop on chemical bonding. You can now:
* combine atomic orbitals into bonding and antibonding molecular orbitals,
* build MO diagrams and explain bond order and the magnetism of O$_2$,
* use crystal field theory to explain the colour and magnetism of transition-metal complexes.

These same computational tools — PySCF and orbital visualisation — are exactly what practising computational chemists use to design new molecules and materials.